# Avalanche day predictor: Davos

**Question:** how likely are natural avalanches around Davos on a given day, and how does a simple model compare with the official SLF forecasters?

**Plan**
1. Data: avalanche observations (SLF, 1999-2019) + daily weather (Open-Meteo)
2. Features: new snow, wind, temperature, sun, ...
3. Targets: avalanche day / dry avalanche day / wet avalanche day (+ was yesterday an avalanche day?)
4. Model: one logistic regression for dry and one for wet avalanches, combined into one risk
5. Test on winters the model never saw, and compare with the forecasters

In [ ]:
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
BLUE, ORANGE, GREEN, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"

## 1. Data

**Avalanches:** SLF, *Snow avalanche data Davos, 1999-2019* ([doi:10.16904/envidat.134](https://www.envidat.ch/dataset/snow-avalanche-data-davos)). One row per winter day: how many avalanches of each size, type and trigger, plus the official danger level (1-5). Download it manually into `data/`.

In [ ]:
activity = pd.read_csv("data/avalanche_activity_data.csv", sep=";", parse_dates=["date"])  # the SLF files use ";"
print(f"{len(activity)} winter days, {activity['date'].min().date()} to {activity['date'].max().date()}")

**Weather:** Open-Meteo archive (free, no API key). This is *reanalysis*: a weather model of the past on a ~10 km grid, not station measurements. We ask for 2500 m, because avalanches start high up (Open-Meteo adjusts temperature to that height). Downloaded one winter at a time and saved, so it only runs once.

In [ ]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_FILE = "data/weather_daily.csv"
DAILY_VARIABLES = [
    "snowfall_sum", "precipitation_sum", "rain_sum",
    "temperature_2m_max", "temperature_2m_min",
    "wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant",
    "shortwave_radiation_sum",
]


def fetch_one_winter(winter):
    """Weather for one winter. Winter 2012 = 1 Oct 2011 to 30 Jun 2012 (same as SLF)."""
    params = {
        "latitude": 46.80, "longitude": 9.84, "elevation": 2500,
        "start_date": f"{winter - 1}-10-01", "end_date": f"{winter}-06-30",
        "timezone": "Europe/Zurich",
        "daily": ",".join(DAILY_VARIABLES),
        "hourly": "snow_depth",  # snow depth only exists hourly
    }
    response = requests.get(ARCHIVE_URL, params=params, timeout=60)
    response.raise_for_status()  # stop with a clear error if the request failed
    data = response.json()

    table = pd.DataFrame(data["daily"]).rename(columns={"time": "date"})
    # Hourly snow depth (m) -> daily mean (cm)
    hourly = pd.DataFrame(data["hourly"])
    hourly["date"] = hourly["time"].str[:10]  # "2012-02-01T13:00" -> "2012-02-01"
    table["snow_depth_cm"] = table["date"].map(hourly.groupby("date")["snow_depth"].mean() * 100)
    return table


if os.path.exists(WEATHER_FILE):
    weather = pd.read_csv(WEATHER_FILE)
else:
    winter_tables = []
    for winter in range(1999, 2020):
        print(f"Downloading winter {winter - 1}-{winter} ...")
        winter_tables.append(fetch_one_winter(winter))
    weather = pd.concat(winter_tables, ignore_index=True)
    weather.to_csv(WEATHER_FILE, index=False)

print(f"{len(weather)} days of weather, missing values: {weather.isna().sum().sum()}")

## 2. Features

What causes avalanches, and how we measure it from the weather:

| Cause | Features |
|---|---|
| New snow loads the snowpack | `snow_24h`, `snow_72h`, `snow_7d`, `days_since_snow`, `snow_depth` |
| Wind moves loose snow into slabs | `wind_max`, `wind_gust`, `wind_snow` (wind x new snow), wind direction |
| Weak layers grow when cold air sits over a thin snowpack | `temp_gradient` |
| Warming and sun make the snow wet and weak | `temp_max`, `temp_min`, `temp_change`, `warm_3d`, `sun`, `sun_warm`, `rain` |
| Season | one yes/no column per month |
| Avalanche cycles | `avalanche_yesterday` (added in section 3) |

**No leakage:** every feature only uses today and earlier days, never the future.

In [ ]:
MONTH_COLUMNS = ["month_Oct", "month_Nov", "month_Dec", "month_Jan", "month_Feb",
                 "month_Mar", "month_Apr", "month_May", "month_Jun"]


def count_days_since_big_snowfall(snowfall):
    """Days since the last snowfall of >= 10 cm (0 = today, max 30)."""
    result = []
    days = 30  # start as "long ago"
    for snow_today in snowfall:
        if snow_today >= 10:
            days = 0
        else:
            days = min(days + 1, 30)
        result.append(days)
    return result


def compute_features(weather):
    """One row per consecutive day in, one row of features per day out."""
    weather = weather.sort_values("date").reset_index(drop=True)
    f = pd.DataFrame()
    f["date"] = pd.to_datetime(weather["date"])

    # Snow. rolling(3).sum() = today + the 2 days before
    f["snow_24h"] = weather["snowfall_sum"]
    f["snow_72h"] = weather["snowfall_sum"].rolling(3, min_periods=1).sum()
    f["snow_7d"] = weather["snowfall_sum"].rolling(7, min_periods=1).sum()
    f["days_since_snow"] = count_days_since_big_snowfall(weather["snowfall_sum"])
    f["snow_depth"] = weather["snow_depth_cm"]
    f["rain"] = weather["rain_sum"]

    # Wind. Direction is an angle (359 and 1 are almost the same), so we split it
    # into a north part (cosine) and an east part (sine).
    f["wind_max"] = weather["wind_speed_10m_max"]
    f["wind_gust"] = weather["wind_gusts_10m_max"]
    f["wind_snow"] = f["wind_gust"] * f["snow_72h"]  # wind can only move snow that is lying loose
    angle = np.radians(weather["wind_direction_10m_dominant"])
    f["wind_from_north"] = np.cos(angle)
    f["wind_from_east"] = np.sin(angle)

    # Temperature and sun
    f["temp_max"] = weather["temperature_2m_max"]
    f["temp_min"] = weather["temperature_2m_min"]
    f["temp_change"] = weather["temperature_2m_max"].diff().fillna(0)  # today - yesterday
    # Temperature gradient (C per metre): the ground under the snow stays near 0 C, the
    # surface is near the air temperature. Cold nights over a thin snowpack (14-day average)
    # grow weak, sugary crystals that stay weak for weeks.
    night_cold = (-weather["temperature_2m_min"]).clip(lower=0)
    snow_depth_m = (weather["snow_depth_cm"] / 100).clip(lower=0.1)  # avoid dividing by ~0
    f["temp_gradient"] = night_cold.rolling(14, min_periods=1).mean() / snow_depth_m
    above_zero = weather["temperature_2m_max"].clip(lower=0)
    f["warm_3d"] = above_zero.rolling(3, min_periods=1).sum()
    f["sun"] = weather["shortwave_radiation_sum"]
    f["sun_warm"] = f["sun"] * above_zero  # sunshine only melts snow when it is warm

    # Season: one yes/no column per month, so each month gets its own baseline risk
    # (risk peaks in mid-winter, which a single straight line cannot capture).
    month_number = (f["date"].dt.month - 10) % 12  # Oct = 0, Nov = 1, ..., Jun = 8
    for number, column in enumerate(MONTH_COLUMNS):
        f[column] = (month_number == number).astype(int)
    return f


features = compute_features(weather)
features.describe().T[["mean", "min", "max"]].round(1)

## 3. Targets

- **Avalanche day:** at least one **natural** avalanche of **size 2+** (big enough to bury a person). Natural only, because weather explains when the snowpack fails on its own; skier-triggered avalanches also depend on how many people are out.
- **Dry / wet day:** the same, split by snow type. They have different causes (dry: new snow and wind; wet: warming and sun), so we train one model for each.
- The official **danger level** is kept as the human benchmark, never as a feature.
- **Avalanche yesterday** is used as a feature: avalanches come in cycles, and yesterday's observations are already known today.

In [ ]:
days = pd.DataFrame()
days["date"] = activity["date"]
days["winter"] = activity["year"]  # winter 2012 = 2011-2012
days["danger_level"] = activity["max.danger.corr"]
days["avalanche_day"] = (activity["NATURAL.SIZE_234"] >= 1).astype(int)
dry_count = activity["dry.NATURAL.SIZE2"] + activity["dry.NATURAL.SIZE3"] + activity["dry.NATURAL.SIZE4"]
wet_count = activity["wet.NATURAL.SIZE2"] + activity["wet.NATURAL.SIZE3"] + activity["wet.NATURAL.SIZE4"]
days["dry_day"] = (dry_count >= 1).astype(int)
days["wet_day"] = (wet_count >= 1).astype(int)

# Was yesterday an avalanche day? Look up the day before; if SLF has no record for it, use 0.
yesterday = days[["date", "avalanche_day"]].copy()
yesterday["date"] = yesterday["date"] + pd.Timedelta(days=1)  # move each day one day forward
yesterday = yesterday.rename(columns={"avalanche_day": "avalanche_yesterday"})
days = days.merge(yesterday, on="date", how="left")
days["avalanche_yesterday"] = days["avalanche_yesterday"].fillna(0)

# Join: keep only SLF days. Features were computed on the full weather series first,
# so rolling windows use the real previous days even when SLF skips a day.
data = days.merge(features, on="date", how="inner")
print(f"{len(data)} days. Avalanche days: {data['avalanche_day'].mean():.1%} "
      f"(dry {data['dry_day'].mean():.1%}, wet {data['wet_day'].mean():.1%})")

## 4. Train

- **Split by whole winters, chosen at random:** 15 winters to learn from, 6 to test on. Whole winters, not random days: neighbouring days are almost identical, so random days would put near-copies in both sets. Random winters (instead of "the last 5") also mix early and late years in both sets, so changes over time (e.g. more avalanches recorded after 2015) affect both equally.
- **Logistic regression:** standardise each feature (mean 0, std 1), then find the weights that minimise the log loss. P = 1 / (1 + e^-(b0 + b1 z1 + ...)).
- **Combine:** P(avalanche) = 1 - (1 - P_dry)(1 - P_wet), the chance that at least one type happens.

In [ ]:
DRY_FEATURES = ["snow_24h", "snow_72h", "snow_7d", "days_since_snow", "snow_depth",
                "wind_gust", "wind_snow", "wind_from_north", "wind_from_east",
                "temp_max", "temp_gradient", "avalanche_yesterday"] + MONTH_COLUMNS
WET_FEATURES = ["temp_max", "temp_min", "temp_change", "warm_3d", "sun", "sun_warm",
                "rain", "snow_depth", "snow_24h", "avalanche_yesterday"] + MONTH_COLUMNS

rng = np.random.default_rng(seed=0)  # fixed seed: the same "random" winters every run
all_winters = sorted(data["winter"].unique())
test_winters = sorted(int(w) for w in rng.choice(all_winters, size=6, replace=False))

train = data[~data["winter"].isin(test_winters)].copy()
test = data[data["winter"].isin(test_winters)].copy()
print(f"Test winters: {test_winters}")
print(f"Train: {len(train)} days, avalanche days {train['avalanche_day'].mean():.0%}")
print(f"Test:  {len(test)} days, avalanche days {test['avalanche_day'].mean():.0%}")


def train_logistic(rows, features, target):
    scaler = StandardScaler()
    x = scaler.fit_transform(rows[features])  # mean and std from the TRAINING data only
    model = LogisticRegression(max_iter=1000)
    model.fit(x, rows[target])
    return scaler, model


def predict(scaler, model, rows, features):
    return model.predict_proba(scaler.transform(rows[features]))[:, 1]


dry_scaler, dry_model = train_logistic(train, DRY_FEATURES, "dry_day")
wet_scaler, wet_model = train_logistic(train, WET_FEATURES, "wet_day")

for rows in [train, test]:
    rows["p_dry"] = predict(dry_scaler, dry_model, rows, DRY_FEATURES)
    rows["p_wet"] = predict(wet_scaler, wet_model, rows, WET_FEATURES)
    rows["p_model"] = 1 - (1 - rows["p_dry"]) * (1 - rows["p_wet"])

## 5. Risk levels

Fixed cut-offs (15/35/60%) left the top levels almost empty. Instead, the model uses each level as often as the forecasters used theirs on the training years: low ~ danger level 1, moderate ~ 2, considerable ~ 3, high ~ 4-5.

In [ ]:
cutoffs = {
    "moderate": train["p_model"].quantile((train["danger_level"] <= 1).mean()),
    "considerable": train["p_model"].quantile((train["danger_level"] <= 2).mean()),
    "high": train["p_model"].quantile((train["danger_level"] <= 3).mean()),
}


def risk_level(probability):
    if probability >= cutoffs["high"]:
        return "high"
    if probability >= cutoffs["considerable"]:
        return "considerable"
    if probability >= cutoffs["moderate"]:
        return "moderate"
    return "low"


print("Cut-offs:", {name: f"{value:.0%}" for name, value in cutoffs.items()})

# Do the levels mean something on the TEST winters?
test["level"] = [risk_level(p) for p in test["p_model"]]
check = test.groupby("level")["avalanche_day"].agg(["size", "mean"])
check = check.reindex(["low", "moderate", "considerable", "high"])
check.columns = ["days", "actual avalanche rate"]
check.round(2)

## 6. Model vs forecasters (test winters)

- **AUC:** pick one avalanche day and one quiet day; how often does the method give the avalanche day the higher score? (0.5 = coin flip, 1 = perfect)
- **Hit rate:** share of avalanche days with a warning. **False alarms:** share of warnings where nothing happened.
- A warning = considerable or higher (forecasters: danger level 3 or higher).

In [ ]:
avalanche = test["avalanche_day"] == 1
model_warns = test["p_model"] >= cutoffs["considerable"]
human_warns = test["danger_level"] >= 3

results = pd.DataFrame({
    "method": ["Forecasters (danger level)", "Our model (dry + wet)"],
    "AUC": [roc_auc_score(avalanche, test["danger_level"]), roc_auc_score(avalanche, test["p_model"])],
    "hit rate": [human_warns[avalanche].mean(), model_warns[avalanche].mean()],
    "false alarms": [(~avalanche[human_warns]).mean(), (~avalanche[model_warns]).mean()],
}).round(2)
results.to_csv("outputs/results.csv", index=False)

print(f"Warned on dry avalanche days: model {model_warns[test['dry_day'] == 1].mean():.0%}, "
      f"forecasters {human_warns[test['dry_day'] == 1].mean():.0%}")
print(f"Warned on wet avalanche days: model {model_warns[test['wet_day'] == 1].mean():.0%}, "
      f"forecasters {human_warns[test['wet_day'] == 1].mean():.0%}")
results

**Is this just luck with the test winters?** Repeat the whole split + training 200 times, each time with 6 other random test winters.

In [ ]:
scores = []
for seed in range(200):
    test_set = np.random.default_rng(seed).choice(all_winters, size=6, replace=False)
    tr = data[~data["winter"].isin(test_set)]
    te = data[data["winter"].isin(test_set)]
    d_scaler, d_model = train_logistic(tr, DRY_FEATURES, "dry_day")
    w_scaler, w_model = train_logistic(tr, WET_FEATURES, "wet_day")
    p = 1 - (1 - predict(d_scaler, d_model, te, DRY_FEATURES)) * (1 - predict(w_scaler, w_model, te, WET_FEATURES))
    scores.append({"model": roc_auc_score(te["avalanche_day"], p),
                   "forecasters": roc_auc_score(te["avalanche_day"], te["danger_level"])})
scores = pd.DataFrame(scores)

print(f"Average AUC over 200 splits: model {scores['model'].mean():.3f}, forecasters {scores['forecasters'].mean():.3f}")
print(f"Model better than forecasters in {(scores['model'] > scores['forecasters']).mean():.0%} of splits")

## 7. Plots

**ROC curve:** avalanche days caught vs quiet days wrongly flagged, for every possible cut-off. Higher and further left is better.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for name, score, color in [("Forecasters", test["danger_level"], GREEN), ("Our model", test["p_model"], BLUE)]:
    false_rate, hit_rate, _ = roc_curve(avalanche, score)
    ax.plot(false_rate, hit_rate, color=color, linewidth=2,
            label=f"{name} (AUC {roc_auc_score(avalanche, score):.2f})")
ax.plot([0, 1], [0, 1], color="#999999", linestyle="--", linewidth=1, label="Coin flip (AUC 0.50)")
ax.set_xlabel("Quiet days wrongly flagged")
ax.set_ylabel("Avalanche days caught")
ax.set_title("Test winters")
ax.legend(loc="lower right", frameon=False)
fig.savefig("outputs/roc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

**What drives each model:** features are standardised, so a longer bar = a bigger effect. Careful: features that move together (snow 24h / 72h / 7d, sun / temperature) share the effect, so read related bars together.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, model, features_used, title, color in [
    (axes[0], dry_model, DRY_FEATURES, "Dry avalanche model", BLUE),
    (axes[1], wet_model, WET_FEATURES, "Wet avalanche model", ORANGE),
]:
    weights = pd.Series(model.coef_[0], index=features_used).drop(MONTH_COLUMNS).sort_values()
    ax.barh(weights.index, weights.values, color=color, height=0.6)
    ax.axvline(0, color="#52514e", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Effect of +1 std on log-odds\n(right = more avalanches)")
fig.tight_layout()
fig.savefig("outputs/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

**Calibration:** when the model says 30%, do avalanches happen 30% of the time? Points above the line = the model is too low, below = too high.

In [ ]:
actual, predicted = calibration_curve(avalanche, test["p_model"], n_bins=8, strategy="quantile")
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot(predicted, actual, marker="o", markersize=8, linewidth=2, color=BLUE, label="Our model")
ax.plot([0, 1], [0, 1], color="#999999", linestyle="--", linewidth=1, label="Perfect")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Actual share of avalanche days")
ax.set_title("Calibration, test winters")
ax.legend(frameon=False)
fig.savefig("outputs/calibration.png", dpi=150, bbox_inches="tight")
plt.show()

**One test winter, day by day:** model risk (top, grey lines = risk levels) and the official danger level (bottom).

In [ ]:
example = test.groupby("winter")["avalanche_day"].mean().idxmax()  # the most active test winter
winter = test[test["winter"] == example]
avalanche_days = winter[winter["avalanche_day"] == 1]
fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
top.plot(winter["date"], winter["p_model"] * 100, color=BLUE, linewidth=2, label="Our model: risk")
top.scatter(avalanche_days["date"], [100] * len(avalanche_days), color=RED, s=30, marker="v",
            label="Observed avalanche day")
for value in cutoffs.values():
    top.axhline(value * 100, color="#cccccc", linewidth=1, zorder=0)
top.set_ylabel("Risk (%)")
top.set_ylim(0, 105)
top.set_title(f"Winter {example - 1}-{example}: predicted risk vs. what happened")
top.legend(loc="upper left", bbox_to_anchor=(0, 0.9), frameon=False)
bottom.step(winter["date"], winter["danger_level"], where="mid", color=GREEN, linewidth=2)
bottom.set_ylabel("Official\ndanger level")
bottom.set_yticks([1, 2, 3, 4, 5])
fig.tight_layout()
fig.savefig("outputs/example_winter.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save the model

In [ ]:
joblib.dump({
    "dry_scaler": dry_scaler, "dry_model": dry_model, "dry_features": DRY_FEATURES,
    "wet_scaler": wet_scaler, "wet_model": wet_model, "wet_features": WET_FEATURES,
    "cutoffs": cutoffs,
}, "data/model.joblib")
train.to_csv("data/train_days.csv", index=False)  # for "find similar past days" later
print("Saved data/model.joblib and data/train_days.csv")

## Conclusions

- On unseen winters, the model beats the official forecasters: average AUC **0.74 vs 0.69** over 200 random splits, better in **98%** of them. Knowing whether avalanches happened yesterday helps a lot (avalanches come in cycles).
- **Dry** avalanches follow new snow (last 7 days); **wet** avalanches follow warmth.
- The model catches more **wet** avalanche days than the forecasters, but fewer **dry** ones.

**Limitations**
- Reanalysis weather for one ~10 km grid cell: wind especially is weak (correlation only 0.43 with measured wind at Weissfluhjoch).
- One yes/no per day for the whole region: no slope direction, so wind loading and sun on south slopes cannot be separated.
- About twice as many avalanches were recorded per day after 2015 (more thorough observation). Random winters mix early and late years in train and test, so this affects both equally.
- Tested and dropped: one model per slope direction (N/E/S/W) did not improve the prediction (AUC 0.72 vs 0.73).
- Not a replacement for the official SLF avalanche bulletin.